# Эксперимент Long × Short Consensus

Самодостаточный ноутбук для двух-модельного фильтра сделок. Главная
гипотеза: одна модель часто обманывается на «псевдо-лонгах»
(прогнозы около P(UP)=прайор), а вторая модель, обученная на
**инвертированном** cost-aware таргете, может вето отвергать такие
сигналы.

**Логика:**

| фаза | условие |
|---|---|
| OPEN long | `P_long > T_long` И `P_short < T_short` |
| CLOSE long | `P_short > T_short` И `P_long < T_long` |

Шорты НЕ открываются — short-модель используется только для
кросс-валидации лонга. Sweep по `(T_long, T_short)` дешёвый: MC-
инференс делается один раз, на каждой комбинации — пересчёт
булевых решений + signal-driven backtest.

**Структура:**
1. Окружение, Drive, данные.
2. Long: датасет + модель (загрузить артефакт или обучить).
3. Short: датасет + модель (всегда обучается).
4. MC-инференс обеих моделей на test и val.
5. Consensus-фрейм + 2D threshold sweep + heatmaps.


## 0. Окружение

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    if PROJECT_ROOT.exists():
        print(f'{PROJECT_ROOT} уже существует — подтягиваем latest commits...')
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', REPO_BRANCH],
                       check=True, env=env)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'reset', '--hard',
                        f'origin/{REPO_BRANCH}'], check=True, env=env)
        head = subprocess.run(
            ['git', '-C', str(PROJECT_ROOT), 'rev-parse', '--short', 'HEAD'],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
        print(f'Обновлено до origin/{REPO_BRANCH} (HEAD={head})')
    else:
        subprocess.run(
            ['git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
             f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git', str(PROJECT_ROOT)],
            check=True, env=env,
        )
        print(f'Репозиторий склонирован в {PROJECT_ROOT}')
else:
    PROJECT_ROOT = Path.cwd().parent

PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'IN_COLAB:     {IN_COLAB}')


In [ ]:
if IN_COLAB:
    print('Устанавливаем зависимости...')
    subprocess.run(
        [
            'pip', 'install', '--quiet',
            'torch>=2.2', 'pandas>=2.1', 'pyarrow>=15.0',
            'scikit-learn>=1.4', 'requests>=2.31', 'yfinance>=0.2.40',
            'pydantic>=2.6', 'tqdm>=4.66', 'ipywidgets>=8.0',
        ],
        check=True,
    )
    print('Готово.')
else:
    print('Локальный режим: используются зависимости из текущего окружения.')


In [ ]:
import sys

SRC = PROJECT_ROOT / 'src'
if not (SRC / 'graduate_work' / '__init__.py').exists():
    raise FileNotFoundError(
        f'graduate_work не найден по пути {SRC}/graduate_work. '
        f'Содержимое {PROJECT_ROOT}: {sorted(p.name for p in PROJECT_ROOT.iterdir())}.'
    )
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

import dataclasses
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from graduate_work.config import default_config

cfg = default_config()
print(f'Тикеры ({len(cfg.data.tickers)}):', cfg.data.tickers)
print('Горизонты:', cfg.data.horizons)
print('Окно:', cfg.data.window_size)
print('Mode:', cfg.data.mode)
print('Архитектура:', cfg.model.architecture)


## 1. Подключение Drive и подтяжка данных

Источник истины — `MyDrive/lstm_exchange/data/raw/`. Если по
какому-то тикеру файлов нет — клик в `training_pipeline.ipynb`,
ячейка скачивания, чтобы их докачать.


In [ ]:
import shutil

DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    drive_raw = DRIVE_BASE / 'data' / 'raw'
    if drive_raw.exists():
        shutil.copytree(drive_raw, cfg.paths.data_raw, dirs_exist_ok=True)
        n = sum(1 for _ in cfg.paths.data_raw.rglob('*.csv'))
        print(f'Подтянули с Drive {n} CSV в {cfg.paths.data_raw}')
    else:
        raise FileNotFoundError(
            f'На Drive нет {drive_raw}. Запусти training_pipeline.ipynb '
            'для первоначальной загрузки данных.'
        )


## 2. Long-сторона: датасет + модель

Long-модель пытаемся **загрузить с Drive** (артефакт сохраняется
training_pipeline.ipynb в `MyDrive/lstm_exchange/checkpoints/`).
Если его там нет — обучаем здесь.


In [ ]:
from graduate_work.features import build_dataset

# Long-датасет — то же что и в training_pipeline.
prepared_long = build_dataset(
    cfg.data, cfg.paths,
    persist=True,
    trading_cfg=cfg.trading,
)
print(f'Long  train: {prepared_long.train["x"].shape}')
print(f'Long   val : {prepared_long.val["x"].shape}')
print(f'Long  test : {prepared_long.test["x"].shape}')
print()
print('=== P(UP) per horizon (long) ===')
for split_name, split in (('train', prepared_long.train),
                          ('val',   prepared_long.val),
                          ('test',  prepared_long.test)):
    line = f'  {split_name:>5s}:'
    for j, hz in enumerate(cfg.data.horizons):
        line += f'  h={hz:>2d}={float((split["y"][:, j] >= 0.5).mean()):.3f}'
    print(line)


In [ ]:
from graduate_work.model import build_model
from graduate_work.training import Trainer, mc_predict, set_seed

# Try loading long-model checkpoint from Drive.
DRIVE_CHECKPOINTS = DRIVE_BASE / 'checkpoints'
LOCAL_CHECKPOINTS = cfg.paths.checkpoints

if IN_COLAB and DRIVE_CHECKPOINTS.exists():
    shutil.copytree(DRIVE_CHECKPOINTS, LOCAL_CHECKPOINTS, dirs_exist_ok=True)

artifact_path = LOCAL_CHECKPOINTS / 'model_artifact.pt'
have_artifact = artifact_path.exists()
print(f'Long-checkpoint найден: {have_artifact} ({artifact_path})')


def _train_long_from_scratch():
    set_seed(cfg.training.seed)
    model = build_model(
        input_dim=prepared_long.num_features,
        num_horizons=len(cfg.data.horizons),
        cfg=cfg.model,
    )
    trainer = Trainer(
        model, cfg.training,
        data_cfg=cfg.data,
        trading_cfg=cfg.trading,
    )
    trainer.fit(prepared_long.train, prepared_long.val)
    return model


if have_artifact:
    # Просто инстанцируем ту же архитектуру и грузим веса.
    set_seed(cfg.training.seed)
    long_model = build_model(
        input_dim=prepared_long.num_features,
        num_horizons=len(cfg.data.horizons),
        cfg=cfg.model,
    )
    state = torch.load(artifact_path, map_location='cpu')
    long_model.load_state_dict(state)
    if torch.cuda.is_available():
        long_model = long_model.cuda()
    long_model.eval()
    print('Long-модель загружена из артефакта.')
else:
    print('Артефакт не найден — обучаем long-модель с нуля.')
    long_model = _train_long_from_scratch()


In [ ]:
is_classification = cfg.data.mode == 'classification'

long_mean, long_std = mc_predict(
    long_model, prepared_long.test['x'],
    mc_passes=cfg.training.mc_passes,
    batch_size=cfg.training.batch_size,
    apply_sigmoid=is_classification,
)
long_val_mean, long_val_std = mc_predict(
    long_model, prepared_long.val['x'],
    mc_passes=cfg.training.mc_passes,
    batch_size=cfg.training.batch_size,
    apply_sigmoid=is_classification,
)
print(f'Long MC test: shape={long_mean.shape}, '
      f'prob mean={long_mean.mean():.4f} max={long_mean.max():.4f}')
print(f'Long MC val:  shape={long_val_mean.shape}, '
      f'prob mean={long_val_mean.mean():.4f} max={long_val_mean.max():.4f}')


## 3. Short-сторона: датасет + модель (всегда обучается)

Один и тот же `cfg.model`, но `swap_long_short_labels=True` —
cost-aware-метка теперь предсказывает выгодность ШОРТА.


In [ ]:
short_data_cfg = dataclasses.replace(cfg.data, swap_long_short_labels=True)
prepared_short = build_dataset(
    short_data_cfg, cfg.paths,
    persist=False,                      # не затираем long processed
    trading_cfg=cfg.trading,
)
print(f'Short train: {prepared_short.train["x"].shape}')
print(f'Short  val : {prepared_short.val["x"].shape}')
print(f'Short test : {prepared_short.test["x"].shape}')
print()
print('=== P(SHORT) per horizon ===')
for split_name, split in (('train', prepared_short.train),
                          ('val',   prepared_short.val),
                          ('test',  prepared_short.test)):
    line = f'  {split_name:>5s}:'
    for j, hz in enumerate(cfg.data.horizons):
        line += f'  h={hz:>2d}={float((split["y"][:, j] >= 0.5).mean()):.3f}'
    print(line)


In [ ]:
set_seed(cfg.training.seed + 1)  # разные сиды для long и short
short_model = build_model(
    input_dim=prepared_short.num_features,
    num_horizons=len(cfg.data.horizons),
    cfg=cfg.model,
)
short_trainer = Trainer(
    short_model, cfg.training,
    data_cfg=short_data_cfg,
    trading_cfg=cfg.trading,
)
short_history = short_trainer.fit(prepared_short.train, prepared_short.val)

print()
print('=== ДИАГНОСТИКА SHORT-ОБУЧЕНИЯ ===')
n_ep = len(short_history.train_loss)
if n_ep:
    best_ep = int(np.argmin(short_history.val_loss)) + 1
    best_val = float(min(short_history.val_loss))
    print(f'Эпох: {n_ep}; best epoch: {best_ep} (val_loss={best_val:.5f})')


In [ ]:
short_mean, short_std = mc_predict(
    short_model, prepared_short.test['x'],
    mc_passes=cfg.training.mc_passes,
    batch_size=cfg.training.batch_size,
    apply_sigmoid=is_classification,
)
short_val_mean, short_val_std = mc_predict(
    short_model, prepared_short.val['x'],
    mc_passes=cfg.training.mc_passes,
    batch_size=cfg.training.batch_size,
    apply_sigmoid=is_classification,
)
print(f'Short MC test: shape={short_mean.shape}, '
      f'prob mean={short_mean.mean():.4f} max={short_mean.max():.4f}')
print(f'Short MC val:  shape={short_val_mean.shape}, '
      f'prob mean={short_val_mean.mean():.4f} max={short_val_mean.max():.4f}')

print()
print('=== SHORT MC by horizon (test) ===')
for j, hz in enumerate(cfg.data.horizons):
    m = short_mean[:, j]
    print(f'  h={hz:>2d}: mean={m.mean():.4f} max={m.max():.4f} '
          f'#prob>0.5={int((m > 0.5).sum())} '
          f'({int((m > 0.5).sum())/len(m)*100:.2f}%)')


## 4. Consensus + 2D threshold sweep

Сборка consensus-фрейма (один раз) и прогон сетки `(T_long, T_short)`
с signal-driven бэктестом.


In [ ]:
from graduate_work.backtest import (
    compute_metrics, run_consensus_backtest,
)
from graduate_work.backtest.engine import prices_from_full_frame
from graduate_work.strategy import (
    apply_consensus_thresholds,
    build_consensus_frame,
    build_predictions_frame,
    consensus_summary,
    ConsensusThresholds,
)

# Готовим test_prices (та же логика что в §7 training_pipeline).
full = prepared_long.full_frame.copy()
if not isinstance(full.index, pd.DatetimeIndex):
    full.index = pd.to_datetime(full.index, utc=True)
test_start = pd.to_datetime(min(prepared_long.test['timestamp']), utc=True)
buffer = cfg.data.bar_timedelta * max(cfg.data.horizons)
test_end = pd.to_datetime(max(prepared_long.test['timestamp']), utc=True) + buffer
test_prices = prices_from_full_frame(
    full.loc[(full.index >= test_start) & (full.index <= test_end)],
)
bars_per_year = cfg.data.bars_per_year

# Long и short predictions в long-формате.
long_predictions = build_predictions_frame(
    timestamps=prepared_long.test['timestamp'],
    tickers=prepared_long.test['ticker'],
    mean=long_mean, std=long_std,
    horizons=cfg.data.horizons,
)
short_predictions = build_predictions_frame(
    timestamps=prepared_short.test['timestamp'],
    tickers=prepared_short.test['ticker'],
    mean=short_mean, std=short_std,
    horizons=cfg.data.horizons,
)
consensus_frame = build_consensus_frame(long_predictions, short_predictions)

print(f'Consensus-фрейм: {len(consensus_frame)} строк '
      f'(было long={len(long_predictions)}, short={len(short_predictions)}).')
print()
print('Распределение по argmax-h:')
for h, sub in consensus_frame.groupby('horizon'):
    print(f'  h={int(h):>2d}: n={len(sub):>5d}  '
          f'P_long mean={sub["p_long"].mean():.4f} max={sub["p_long"].max():.4f}  '
          f'P_short mean={sub["p_short"].mean():.4f} max={sub["p_short"].max():.4f}')


In [ ]:
# --- 2D threshold sweep ---
t_long_grid  = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
t_short_grid = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
max_hold_bars = max(cfg.data.horizons)   # потолок hold = 4 часа (48 баров)

sweep_rows = []
sweep_equities = {}

for t_l in t_long_grid:
    for t_s in t_short_grid:
        thr = ConsensusThresholds(t_long=t_l, t_short=t_s)
        decisions = apply_consensus_thresholds(consensus_frame, thr)
        n_open  = int(decisions['open_long'].sum())
        n_close = int(decisions['close_long'].sum())
        if n_open == 0:
            sweep_rows.append({
                't_long': t_l, 't_short': t_s,
                'n_open_signals': 0, 'n_close_signals': n_close,
                'n_trades': 0,
                'final_equity': float(cfg.trading.initial_capital),
                'total_return': 0.0, 'sharpe': 0.0,
                'max_dd': 0.0, 'win_rate': 0.0,
            })
            continue
        bt = run_consensus_backtest(
            decisions, test_prices, cfg.trading,
            max_hold_bars=max_hold_bars,
        )
        m = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
        sweep_rows.append({
            't_long': t_l, 't_short': t_s,
            'n_open_signals': n_open, 'n_close_signals': n_close,
            'n_trades':       int(m['n_trades']),
            'final_equity':   float(m['final_equity']),
            'total_return':   float(m['total_return']),
            'sharpe':         float(m['sharpe']),
            'max_dd':         float(m['max_drawdown']),
            'win_rate':       float(m['win_rate']),
        })
        if not bt.equity.empty:
            sweep_equities[(t_l, t_s)] = bt.equity

consensus_sweep = pd.DataFrame(sweep_rows)
print('=== CONSENSUS THRESHOLD SWEEP ===')
print(consensus_sweep.to_string(
    index=False,
    float_format=lambda x: f'{x:.4f}',
))


In [ ]:
# --- Heatmaps + лучшая комбинация ---
def _pivot(metric: str) -> pd.DataFrame:
    return consensus_sweep.pivot(index='t_long', columns='t_short', values=metric)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, metric, fmt, title in [
    (axes[0], 'sharpe',       '.2f', 'Sharpe'),
    (axes[1], 'total_return', '.3f', 'Total return'),
    (axes[2], 'n_trades',     '.0f', '# Trades'),
    (axes[3], 'win_rate',     '.2f', 'Win rate'),
]:
    grid = _pivot(metric)
    im = ax.imshow(grid.values, aspect='auto', cmap='RdYlGn',
                   vmin=grid.values.min(), vmax=grid.values.max())
    ax.set_xticks(range(len(grid.columns))); ax.set_xticklabels([f'{x:.2f}' for x in grid.columns])
    ax.set_yticks(range(len(grid.index)));   ax.set_yticklabels([f'{x:.2f}' for x in grid.index])
    ax.set_xlabel('T_short'); ax.set_ylabel('T_long')
    ax.set_title(title)
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            ax.text(j, i, f'{grid.values[i, j]:{fmt}}',
                    ha='center', va='center', fontsize=8, color='black')
    fig.colorbar(im, ax=ax, fraction=0.04)
fig.tight_layout(); plt.show()

profitable = consensus_sweep[consensus_sweep['n_trades'] > 0]
if not profitable.empty:
    best_sharpe = profitable.loc[profitable['sharpe'].idxmax()]
    best_return = profitable.loc[profitable['total_return'].idxmax()]
    print()
    print(f'Best by Sharpe:        T_long={best_sharpe["t_long"]:.2f}  '
          f'T_short={best_sharpe["t_short"]:.2f}  '
          f'Sharpe={best_sharpe["sharpe"]:.4f}  '
          f'return={best_sharpe["total_return"]*100:.2f}%  '
          f'n_trades={int(best_sharpe["n_trades"])}')
    print(f'Best by total_return:  T_long={best_return["t_long"]:.2f}  '
          f'T_short={best_return["t_short"]:.2f}  '
          f'Sharpe={best_return["sharpe"]:.4f}  '
          f'return={best_return["total_return"]*100:.2f}%  '
          f'n_trades={int(best_return["n_trades"])}')

    # Equity лучших комбинаций.
    fig, ax = plt.subplots(figsize=(11, 4.5))
    keys_to_plot = {
        'best Sharpe': (best_sharpe['t_long'], best_sharpe['t_short']),
        'best return': (best_return['t_long'], best_return['t_short']),
    }
    colors = {'best Sharpe': '#9467bd', 'best return': '#d62728'}
    for label, key in keys_to_plot.items():
        if key in sweep_equities:
            eq = sweep_equities[key]
            ax.plot(eq.index, eq.values, color=colors[label], linewidth=1.4,
                    label=f'{label}: T_l={key[0]:.2f} T_s={key[1]:.2f}')
    ax.axhline(cfg.trading.initial_capital, color='gray', linestyle=':', alpha=0.6)
    ax.set_title('Лучшие consensus-комбинации')
    ax.set_xlabel('дата'); ax.set_ylabel('капитал, RUB')
    ax.legend(); fig.autofmt_xdate(); fig.tight_layout(); plt.show()
else:
    print()
    print('Ни одна (T_long, T_short) комбинация не дала сделок. '
          'Возможные причины: (1) обе модели в prediction collapse, '
          '(2) max(P_long) и max(P_short) ниже даже минимального порога 0.50. '
          'Проверь диагностику MC выше.')


## Выводы

- **Консервативная зона** (T_long и T_short оба высокие, ≥ 0.65) — мало сделок,
  но если модели реально нашли edge, win_rate высокий. Это «всё подтверждено
  обеими моделями».
- **Агрессивная зона** (T_long ~ 0.50, T_short ~ 0.50) — много сделок,
  но фильтр почти не работает: short-модель отвергает только когда уверена.
- **Асимметричный угол** (T_long высокий, T_short низкий) — открываем
  редко, но любое колебание шорта закрывает позицию. Подходит для
  волатильных режимов.
- **Лучшая комбинация** обычно лежит на диагонали где P_long и
  P_short сравнимо высокие — это «дисагримент пары».

Для защиты ВКР: heatmap демонстрирует, что результат **не выбран
ad-hoc** — это полная картина по всему пространству порогов, и
оптимум стабилен в небольшой области. Это сильнее, чем один backtest.
